<a href="https://colab.research.google.com/github/sathwikayasa/Healthcare-revenue-cycle-analysis/blob/main/01_generate_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# 1. Settings
# ---------------------------------------------------------------
SEED = 42                       # same seed = same dataset every run
N_CLAIMS = 50_000
START = pd.Timestamp("2024-07-01")
SNAPSHOT = pd.Timestamp("2026-06-30")   # "today" for the analysis

rng = np.random.default_rng(SEED)

# ---------------------------------------------------------------
# 2. Lookup tables
# ---------------------------------------------------------------
departments = pd.DataFrame({
    "department_id":   [1, 2, 3, 4, 5, 6, 7, 8],
    "department_name": ["Emergency", "Primary Care", "Cardiology", "Orthopedics",
                        "Radiology", "Laboratory", "Oncology", "General Surgery"],
    "share":           [0.18, 0.22, 0.10, 0.09, 0.14, 0.15, 0.05, 0.07],
    "avg_charge":      [2400, 320, 4200, 6800, 950, 180, 9500, 12500],
    "avg_charge_lag":  [6.0, 1.5, 2.0, 2.5, 1.5, 1.0, 2.0, 3.0],   # days from service to charge entry
    "denial_multiplier": [1.25, 1.0, 1.0, 1.30, 1.0, 1.0, 1.0, 1.30],
})

payers = pd.DataFrame({
    "payer_id":      [1, 2, 3, 4, 5, 6],
    "payer_name":    ["Medicare", "Medicaid", "Northstar Health",
                      "Summit Mutual", "Lakeview Insurance", "Self-Pay"],
    "payer_type":    ["Government", "Government", "Commercial",
                      "Commercial", "Commercial", "Self-Pay"],
    "share":         [0.30, 0.17, 0.20, 0.14, 0.11, 0.08],
    "contract_rate": [0.40, 0.32, 0.62, 0.58, 0.60, 0.55],   # allowed amount as a share of billed
    "base_denial":   [0.06, 0.09, 0.06, 0.17, 0.07, 0.00],
    "median_days_to_pay": [16, 48, 24, 33, 27, 60],
    "patient_share": [0.15, 0.02, 0.20, 0.20, 0.20, 1.00],   # part of allowed amount owed by the patient
    "patient_collection": [0.75, 0.50, 0.65, 0.65, 0.65, 0.45],
})

providers = pd.DataFrame({
    "provider_id": np.arange(1, 41),
    "department_id": np.repeat(departments["department_id"].values, 5),
})
providers["provider_name"] = "Provider " + providers["provider_id"].astype(str).str.zfill(3)

DENIAL_REASONS = ["Prior authorization missing", "Eligibility or coverage", "Coding error",
                  "Missing documentation", "Medical necessity", "Timely filing", "Duplicate claim"]
REASON_DEFAULT   = [0.22, 0.20, 0.20, 0.15, 0.12, 0.06, 0.05]
REASON_SUMMIT    = [0.45, 0.13, 0.14, 0.11, 0.10, 0.04, 0.03]   # Summit Mutual: prior authorization heavy
REASON_EMERGENCY = [0.08, 0.40, 0.18, 0.14, 0.10, 0.06, 0.04]   # Emergency: eligibility heavy
APPEAL_PROB   = dict(zip(DENIAL_REASONS, [0.70, 0.65, 0.75, 0.75, 0.60, 0.20, 0.15]))
OVERTURN_PROB = dict(zip(DENIAL_REASONS, [0.55, 0.45, 0.70, 0.75, 0.40, 0.10, 0.05]))

# ---------------------------------------------------------------
# 3. Build the claims
# ---------------------------------------------------------------
n = N_CLAIMS
span_days = (SNAPSHOT - START).days

claims = pd.DataFrame({"claim_id": ["CLM" + str(i).zfill(6) for i in range(1, n + 1)]})
claims["patient_id"] = ["PAT" + str(i).zfill(5) for i in rng.integers(1, 18_001, n)]

# Service dates: volume grows slightly over the two years
t = rng.power(1.15, n)                      # 0 = start of period, 1 = snapshot date
claims["service_date"] = START + pd.to_timedelta((t * span_days).astype(int), unit="D")

claims["department_id"] = rng.choice(departments["department_id"], n, p=departments["share"])
claims["payer_id"] = rng.choice(payers["payer_id"], n, p=payers["share"])

d = (claims.merge(departments, on="department_id", how="left")
           .merge(payers, on="payer_id", how="left", suffixes=("_dept", "_payer")))

# Each claim gets one of the five providers in its department
claims["provider_id"] = (claims["department_id"] - 1) * 5 + rng.integers(1, 6, n)

# --- Money -------------------------------------------------------
sigma = 0.6
billed = rng.lognormal(np.log(d["avg_charge"]) - sigma**2 / 2, sigma)
claims["billed_amount"] = billed.round(2)
allowed = billed * d["contract_rate"] * rng.uniform(0.95, 1.05, n)
claims["allowed_amount"] = allowed.round(2)
claims["contractual_adjustment"] = (claims["billed_amount"] - claims["allowed_amount"]).round(2)
patient_resp = (claims["allowed_amount"] * d["patient_share"]).round(2)

# --- Dates before the payer responds -----------------------------
charge_lag = rng.poisson(d["avg_charge_lag"])
claims["charge_entry_date"] = claims["service_date"] + pd.to_timedelta(charge_lag, unit="D")

is_self_pay = (d["payer_type"] == "Self-Pay").values

# --- Denials -----------------------------------------------------
# Denial risk = payer baseline x department effect x a gradual upward trend
p_denial = (d["base_denial"] * d["denial_multiplier"] * (1 + 0.25 * t)).clip(0, 0.6)
denied = rng.random(n) < p_denial

reason = np.full(n, None, dtype=object)
is_summit = (d["payer_name"] == "Summit Mutual").values
is_er = (d["department_name"] == "Emergency").values
for mask, probs in [(denied & is_summit, REASON_SUMMIT),
                    (denied & is_er & ~is_summit, REASON_EMERGENCY),
                    (denied & ~is_er & ~is_summit, REASON_DEFAULT)]:
    reason[mask] = rng.choice(DENIAL_REASONS, mask.sum(), p=probs)

# Clean claim = accepted on first submission with no rework.
# Denied claims are never clean; 4% of the rest are rejected up front and resubmitted.
rejected_first = (~denied) & (~is_self_pay) & (rng.random(n) < 0.04)
claims["clean_claim_flag"] = np.where(denied | rejected_first, 0, 1)
claims.loc[is_self_pay, "clean_claim_flag"] = 1

submit_lag = rng.integers(1, 4, n) + np.where(rejected_first, 7, 0)
claims["submission_date"] = claims["charge_entry_date"] + pd.to_timedelta(submit_lag, unit="D")

# --- Payer decision ----------------------------------------------
days_to_decision = np.clip(rng.lognormal(np.log(d["median_days_to_pay"]), 0.45).round().astype(int), 3, None)
decision_date = claims["submission_date"] + pd.to_timedelta(days_to_decision, unit="D")

# 3% of insurer claims get no response at all and sit in A/R
stuck = (~denied) & (~is_self_pay) & (rng.random(n) < 0.03)

# --- Appeals -----------------------------------------------------
reason_s = pd.Series(reason)
appealed = denied & (rng.random(n) < reason_s.map(APPEAL_PROB).fillna(0).values)
overturned = appealed & (rng.random(n) < reason_s.map(OVERTURN_PROB).fillna(0).values)
appeal_days = rng.lognormal(np.log(45), 0.4, n).round().astype(int)
appeal_resolved_date = decision_date + pd.to_timedelta(appeal_days, unit="D")

# --- Status as of the snapshot date ------------------------------
decided = (decision_date <= SNAPSHOT).values & ~stuck
appeal_done = (appeal_resolved_date <= SNAPSHOT).values

status = np.full(n, "Pending", dtype=object)
status[decided & ~denied] = "Paid"
status[decided & denied & ~appealed] = "Denied"
status[decided & appealed & ~appeal_done] = "In Appeal"
status[decided & appealed & appeal_done & overturned] = "Paid"
status[decided & appealed & appeal_done & ~overturned] = "Denied"

# Self-pay: 45% pay their bill, the rest go to bad debt after 120 days
sp_pays = rng.random(n) < 0.45
sp_age = (SNAPSHOT - claims["submission_date"]).dt.days.values
status[is_self_pay & decided & ~sp_pays & (sp_age > 120)] = "Written Off"
status[is_self_pay & decided & ~sp_pays & (sp_age <= 120)] = "Pending"
claims["claim_status"] = status

was_denied = decided & denied            # denial is only known once the payer has responded
claims["denial_flag"] = was_denied.astype(int)
claims["denial_date"] = decision_date.where(was_denied)
claims["denial_reason"] = reason_s.where(was_denied)
claims["appeal_flag"] = (was_denied & appealed).astype(int)
outcome = np.full(n, None, dtype=object)
outcome[was_denied & appealed & ~appeal_done] = "Pending"
outcome[was_denied & appealed & appeal_done & overturned] = "Overturned"
outcome[was_denied & appealed & appeal_done & ~overturned] = "Upheld"
claims["appeal_outcome"] = outcome

paid = status == "Paid"
payment_date = decision_date.where(~(denied & overturned), appeal_resolved_date)
claims["payment_date"] = payment_date.where(paid)

# --- Payments ----------------------------------------------------
claims["patient_responsibility"] = patient_resp
payer_paid = np.where(paid & ~is_self_pay, claims["allowed_amount"] - patient_resp, 0.0)
claims["payer_paid_amount"] = payer_paid.round(2)

# Patients pay some share of what they owe, but only after the insurer has paid
frac = rng.beta(5 * d["patient_collection"], 5 * (1 - d["patient_collection"]))
frac = np.where(is_self_pay, 1.0, frac)
days_since_payment = (SNAPSHOT - payment_date).dt.days.values
patient_billed = paid & (days_since_payment >= 30)
claims["patient_paid_amount"] = np.where(patient_billed, patient_resp * frac, 0.0).round(2)
claims["total_paid_amount"] = (claims["payer_paid_amount"] + claims["patient_paid_amount"]).round(2)

# --- What is still owed, and what has been given up on -----------
patient_gap = (patient_resp - claims["patient_paid_amount"]).clip(lower=0)
old_patient_balance = paid & (days_since_payment > 120)

outstanding = np.zeros(n)
open_claim = np.isin(status, ["Pending", "In Appeal"])
outstanding[open_claim] = claims["allowed_amount"].values[open_claim]
recent = paid & ~old_patient_balance
outstanding[recent] = patient_gap.values[recent]
claims["outstanding_balance"] = outstanding.round(2)

write_off = np.zeros(n)
lost = np.isin(status, ["Denied", "Written Off"])
write_off[lost] = claims["allowed_amount"].values[lost]
write_off[old_patient_balance] = patient_gap.values[old_patient_balance]
claims["write_off_amount"] = write_off.round(2)

# ---------------------------------------------------------------
# 4. Tidy up and save
# ---------------------------------------------------------------
claims = claims[[
    "claim_id", "patient_id", "provider_id", "department_id", "payer_id",
    "service_date", "charge_entry_date", "submission_date",
    "billed_amount", "contractual_adjustment", "allowed_amount",
    "clean_claim_flag", "claim_status",
    "denial_flag", "denial_date", "denial_reason", "appeal_flag", "appeal_outcome",
    "payment_date", "payer_paid_amount", "patient_responsibility", "patient_paid_amount",
    "total_paid_amount", "outstanding_balance", "write_off_amount",
]].sort_values("service_date").reset_index(drop=True)

claims.to_csv("claims.csv", index=False)
payers[["payer_id", "payer_name", "payer_type"]].to_csv("payers.csv", index=False)
departments[["department_id", "department_name"]].to_csv("departments.csv", index=False)
providers[["provider_id", "provider_name", "department_id"]].to_csv("providers.csv", index=False)

# ---------------------------------------------------------------
# 5. Quick check: print the headline numbers
# ---------------------------------------------------------------
insurer = claims["payer_id"] != 6
adjudicated = insurer & claims["claim_status"].isin(["Paid", "Denied", "In Appeal"])
closed = claims["claim_status"].isin(["Paid", "Denied", "Written Off"])
last_90 = claims["service_date"] > SNAPSHOT - pd.Timedelta(days=90)

print(f"Claims:               {len(claims):,}")
print(f"Total billed:         ${claims['billed_amount'].sum():,.0f}")
print(f"Total collected:      ${claims['total_paid_amount'].sum():,.0f}")
print(f"Clean claim rate:     {claims.loc[insurer, 'clean_claim_flag'].mean():.1%}")
print(f"Denial rate:          {claims.loc[adjudicated, 'denial_flag'].mean():.1%}")
print(f"Net collection rate:  {claims.loc[closed, 'total_paid_amount'].sum() / claims.loc[closed, 'allowed_amount'].sum():.1%}")
print(f"Days in A/R:          {claims['outstanding_balance'].sum() / (claims.loc[last_90, 'allowed_amount'].sum() / 90):.1f}")
print("\nClaims by status:")
print(claims["claim_status"].value_counts().to_string())

Claims:               50,000
Total billed:         $153,435,646
Total collected:      $57,129,695
Clean claim rate:     86.1%
Denial rate:          10.5%
Net collection rate:  83.9%
Days in A/R:          67.4

Claims by status:
claim_status
Paid           41049
Pending         4235
Denied          2733
Written Off     1754
In Appeal        229
